# 08. Search time against storage size

Slide 20 commits us to measuring search time against storage size for both schemes, with $N = 3, 5, 7$
servers. This notebook does that at PARAM I with threshold $t = 2$, for $10^2, 10^3, 10^4$ and $10^5$
files per user, spread evenly over the servers. Each run has $c_w = 3$ real matching files, one each on
$S_1, S_2, S_3$, uploaded through the real Outsource of each scheme. The rest is filler (below).

For each run it reports the slowest server, the sum over the servers, and the user's time:

- **Paper's scheme.** At each server: R2, `retrieve_scan`. At the user: R1 and R3.
- **Counter index.** At each server: R2, `label_search`. At the user: R1, R3, the R4 requests and R5.
  The servers' R4 `fetch_files` is timed separately, as `S_i fetch`, and is not part of the search time.

How each timing is taken, as in `05_costs.ipynb`:

- The kernel is pinned to CPU 2 when the machine has one, and the next cell prints whether it was.
- `bench.measure` runs one warmup and then 5 timed runs, and we print the median and the min-max range.
- Every run, the warmup included, checks its own result, and the last cell prints the failure count.
- Each run starts from a fresh Login (not timed), so the user derives $dsk$ or $K_w$ and the labels
  again, as the first search of a session does.

**Filler.** Real files through Outsource would cost a PRF chain and a signature each, too slow for $10^5$
files. `bench.fill_qpase_storage` and `bench.fill_counter_index` write filler straight into server state.
A filler $Ct$ is AES-256-GCM under its own fresh random key and has the length of a real $Ct$, so
$Dec(dsk, Ct)$ fails at the tag exactly as for another keyword's item. A filler row is a random 32-byte
label with a random entry of a real entry's length.

**What we expect before measuring.** The paper's R2 tries $dsk$ on every stored item, so it should grow
linearly with storage. The counter index does $c_w$ lookups at every size. A failed AES-GCM check is cheap,
and the paper's R2 also runs one PRF chain per matching item, so the growth may only show at the larger
sizes. The tables report what we measured.

The full sweep takes a while: 3 server counts × 4 sizes × 2 schemes × 6 runs, each run with its own Login.
At $10^5$ files the servers hold $10^5$ full-size filler $Ct$ in memory, for one $N$ at a time.

In [1]:
import os
import sys

numpy_before_pinning = "numpy" in sys.modules

from ris_sem_project.bench import PIN_CPU, cpu_model, cpu_summary, pin_to_cpu, snapshot

pin = pin_to_cpu(PIN_CPU)

import importlib.metadata
import platform

load_start = snapshot()
print(f"CPU: {cpu_model()}, {os.cpu_count()} logical CPUs")
print(f"kernel main thread affinity: {list(pin.affinity)} (numpy imported before pinning: {numpy_before_pinning})")
print(pin.describe())
print(cpu_summary(PIN_CPU))
print(f"Python {platform.python_version()}, numpy {importlib.metadata.version('numpy')}, "
      f"dilithium-py {importlib.metadata.version('dilithium-py')}, cryptography {importlib.metadata.version('cryptography')}")

CPU: 12th Gen Intel(R) Core(TM) i5-12450H, 12 logical CPUs
kernel main thread affinity: [2] (numpy imported before pinning: False)
pinned the kernel main thread to CPU 2
CPU 2: max unreadable, governor unreadable, hyperthread siblings 2-3
Python 3.14.10, numpy 2.5.3, dilithium-py 1.4.0, cryptography 50.0.2


## Setup and the trials

`prepare(N)` builds $N$ `CounterIndexServer`s with room for $10^5$ rows, registers alice, logs her in and
uploads the 3 matching files under `medical` through both schemes' Outsource. `fill_to` then tops every
server up to its even share of the storage size in both schemes. The 3 real files count towards the size.

A trial makes the same party calls, in the same order, as `protocol.retrieve` or `protocol.ci_retrieve`, and
wraps each in `clock(party, method, *args)`. It returns whether the result was correct, and the number of
items the servers scanned (paper) or the lookups each server did (counter index).

In [2]:
import statistics

from ris_sem_project import protocol
from ris_sem_project.bench import (
    busy_summary, current_affinity, duration, fill_counter_index, fill_qpase_storage, measure, spread, spread_evenly,
)
from ris_sem_project.display import table
from ris_sem_project.params import PARAM_I, setup
from ris_sem_project.parties.counter_index_server import CounterIndexServer
from ris_sem_project.parties.counter_index_user import CounterIndexUser
from ris_sem_project.parties.qpase_user import QPASEUser

PASSWORD = "correct horse battery staple"
QUERY = "medical"
SIZES = (100, 1_000, 10_000, 100_000)
SERVER_COUNTS = (3, 5, 7)
T = 2
RUNS = 5
MATCHING = (
    (1, b"Blood test results, Riverside Lab, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL cholesterol 2.9 mmol/L, HbA1c 5.2%."),
    (2, b"Physiotherapy referral from Dr. Okafor. Lumbar MRI shows a mild disc bulge at L4-L5 with no nerve compression."),
    (3, b"Vaccination record: influenza 2026/27 dose given 2 Oct 2026, batch FL2611, left arm. Tetanus booster due 2031."),
)
EXPECTED = [d for _, d in MATCHING]


def prepare(N):
    """N servers, alice registered and logged in, the 3 matching files on S_1, S_2, S_3 in both schemes."""
    pp = setup(PARAM_I, N=N, t=T, mu=1000)
    servers = protocol.setup_servers(pp, capacity=max(SIZES), server_type=CounterIndexServer)
    alice = CounterIndexUser(pp, "alice", PASSWORD)
    protocol.register(alice, servers)
    protocol.ci_login(alice, servers, (1, 2))
    paper = [protocol.outsource(alice, servers[i], QUERY, d, i).request for i, d in MATCHING]
    indexed = [protocol.ci_outsource(alice, servers, i, QUERY, d, i).request for i, d in MATCHING]
    return pp, servers, paper, indexed


def fill_to(servers, size, ct_size, entry_size, data_ct):
    """Top every server up to its even share of `size` items (paper) and rows (counter index)."""
    for (i, server), share in zip(sorted(servers.items()), spread_evenly(size, len(servers))):
        held = len(server.storage.get("alice", []))
        fill_qpase_storage(server, "alice", share - held, ct_size, i * 1_000_000 + held, data_ct)
        fill_counter_index(server, "alice", share - len(server.index.get("alice", {})), entry_size)


def paper_trial(pp, servers, size):
    def trial(clock):
        user = QPASEUser(pp, "alice", PASSWORD)
        protocol.login(user, servers, (1, 2))
        request = clock("U", user.retrieve_request, [QUERY])
        scans = {i: clock(f"S_{i}", server.retrieve_scan, request) for i, server in servers.items()}
        opened = clock("U", user.retrieve_open, [QUERY], [scan.response for scan in scans.values()])
        scanned = sum(len(scan.items) for scan in scans.values())
        return [o.data for o in opened] == EXPECTED and scanned == size, [scanned]
    return trial


def indexed_trial(pp, servers):
    def trial(clock):
        user = CounterIndexUser(pp, "alice", PASSWORD)
        protocol.ci_login(user, servers, (1, 2))
        request = clock("U", user.label_search_request, QUERY)
        searches = {i: clock(f"S_{i}", server.label_search, request) for i, server in servers.items()}
        hits = clock("U", user.open_hits, QUERY, [search.response for search in searches.values()])
        wanted = {}
        for hit in hits:
            wanted.setdefault(hit.server_id, []).append(hit.item_id)
        responses = []
        for i, ids in sorted(wanted.items()):
            fetch = clock("U", user.fetch_request, sorted(ids))
            responses.append(clock(f"S_{i} fetch", servers[i].fetch_files, fetch))
        files = clock("U", user.open_files, hits, responses)
        lookups = [search.lookups for search in searches.values()]
        return files == EXPECTED and lookups == [len(EXPECTED)] * len(servers), lookups
    return trial

## The sweep

For each $N$, one set of servers is filled up size by size, and both schemes are measured at each size.
Each line prints what every server holds and the medians, and the tables below give the ranges.

In [3]:
results, held, real_sizes = {}, {}, {}
for N in SERVER_COUNTS:
    pp, servers, paper, indexed = prepare(N)
    real_sizes[N] = (len(paper[0].item.index_ct), len(indexed[0].entry))
    data_ct = os.urandom(len(paper[0].item.data_ct))
    for size in SIZES:
        fill_to(servers, size, *real_sizes[N], data_ct)
        held[(N, size)] = [(len(s.storage["alice"]), len(s.index["alice"])) for s in servers.values()]
        results[(N, size, "paper")] = measure(paper_trial(pp, servers, size), RUNS)
        results[(N, size, "counter index")] = measure(indexed_trial(pp, servers), RUNS)
        medians = {
            scheme: duration(statistics.median(max(run[f"S_{i}"] for i in servers) for run in results[(N, size, scheme)].runs))
            for scheme in ("paper", "counter index")
        }
        print(f"N = {N}, {size} files: (items, rows) per server {held[(N, size)]}; slowest server median: paper {medians['paper']}, counter index {medians['counter index']}")
    del servers, paper, indexed
print(f"real Ct and entry sizes the filler copies: {sorted(set(real_sizes.values()))} B")

N = 3, 100 files: (items, rows) per server [(34, 34), (33, 33), (33, 33)]; slowest server median: paper 1.11 s, counter index 7.36 ms


N = 3, 1000 files: (items, rows) per server [(334, 334), (333, 333), (333, 333)]; slowest server median: paper 961 ms, counter index 6.23 ms


N = 3, 10000 files: (items, rows) per server [(3334, 3334), (3333, 3333), (3333, 3333)]; slowest server median: paper 976 ms, counter index 7.08 ms


N = 3, 100000 files: (items, rows) per server [(33334, 33334), (33333, 33333), (33333, 33333)]; slowest server median: paper 1.19 s, counter index 6.82 ms


N = 5, 100 files: (items, rows) per server [(20, 20), (20, 20), (20, 20), (20, 20), (20, 20)]; slowest server median: paper 924 ms, counter index 7.15 ms


N = 5, 1000 files: (items, rows) per server [(200, 200), (200, 200), (200, 200), (200, 200), (200, 200)]; slowest server median: paper 946 ms, counter index 7.15 ms


... (2 outputs left out)

N = 7, 100 files: (items, rows) per server [(15, 15), (15, 15), (14, 14), (14, 14), (14, 14), (14, 14), (14, 14)]; slowest server median: paper 907 ms, counter index 8.86 ms


N = 7, 1000 files: (items, rows) per server [(143, 143), (143, 143), (143, 143), (143, 143), (143, 143), (143, 143), (142, 142)]; slowest server median: paper 871 ms, counter index 7.62 ms


N = 7, 10000 files: (items, rows) per server [(1429, 1429), (1429, 1429), (1429, 1429), (1429, 1429), (1428, 1428), (1428, 1428), (1428, 1428)]; slowest server median: paper 896 ms, counter index 7.29 ms


N = 7, 100000 files: (items, rows) per server [(14286, 14286), (14286, 14286), (14286, 14286), (14286, 14286), (14286, 14286), (14285, 14285), (14285, 14285)]; slowest server median: paper 961 ms, counter index 6.95 ms
real Ct and entry sizes the filler copies: [(4256, 108)] B


## Results

Per run, "slowest server" is the largest search time over the $N$ servers and "sum over servers" adds
all $N$. Each cell gives the median over the 5 timed runs and the min-max range.

In [4]:
def per_run(measured, N, combine):
    return [combine(run[f"S_{i}"] for i in range(1, N + 1)) for run in measured.runs]


rows, lines = [], []
for (N, size, scheme), measured in results.items():
    slowest, total, user = per_run(measured, N, max), per_run(measured, N, sum), measured.samples("U")
    rows.append((
        N, size, scheme,
        f"{duration(statistics.median(slowest))} ({spread(slowest)})",
        f"{duration(statistics.median(total))} ({spread(total)})",
        f"{duration(statistics.median(user))} ({spread(user)})",
        measured.failures,
    ))
    lines.append(f"N = {N}, {size} files, {scheme}: slowest server median {duration(statistics.median(slowest))}, sum over servers median {duration(statistics.median(total))}, U median {duration(statistics.median(user))}")
print(table(("N", "files", "scheme", "slowest server: median (range)", "sum over servers: median (range)", "U: median (range)", "failures"), rows))
print()
print("\n".join(lines))

N  files   scheme         slowest server: median (range)  sum over servers: median (range)  U: median (range)          failures
-  ------  -------------  ------------------------------  --------------------------------  -------------------------  --------
3  100     paper          1.11 s (1.05 s to 1.25 s)       3.27 s (3.10 s to 3.54 s)         4.45 s (4.23 s to 4.70 s)  0
3  100     counter index  7.36 ms (6.43 ms to 7.69 ms)    21.4 ms (18.6 ms to 22.0 ms)      4.76 s (4.65 s to 4.91 s)  0
3  1000    paper          961 ms (928 ms to 1.12 s)       2.82 s (2.67 s to 3.29 s)         3.89 s (3.45 s to 4.12 s)  0
3  1000    counter index  6.23 ms (5.89 ms to 9.95 ms)    18.2 ms (17.1 ms to 27.4 ms)      3.80 s (3.52 s to 4.27 s)  0
3  10000   paper          976 ms (885 ms to 1.04 s)       2.80 s (2.61 s to 2.96 s)         3.55 s (3.41 s to 3.84 s)  0
3  10000   counter index  7.08 ms (6.18 ms to 9.49 ms)    18.8 ms (18.2 ms to 25.9 ms)      3.73 s (3.61 s to 3.94 s)  0
3  100000  paper  

## Crossover

For each $N$ and size, the ratio of the paper's median search time to the counter index's, for the slowest
server and for the sum over servers. A ratio above 1 means the counter index searched faster. The last
lines give, for each $N$, the smallest measured size from which the counter index stays faster.

In [5]:
ratio_rows, crossover = [], {}
for N in SERVER_COUNTS:
    faster = []
    for size in SIZES:
        paper, indexed = results[(N, size, "paper")], results[(N, size, "counter index")]
        slowest = statistics.median(per_run(paper, N, max)) / statistics.median(per_run(indexed, N, max))
        total = statistics.median(per_run(paper, N, sum)) / statistics.median(per_run(indexed, N, sum))
        ratio_rows.append((N, size, f"{slowest:.1f}", f"{total:.1f}"))
        faster.append(total > 1)
    later = [size for k, size in enumerate(SIZES) if all(faster[k:])]
    crossover[N] = later[0] if later else None
print(table(("N", "files", "paper / counter index, slowest server", "paper / counter index, sum over servers"), ratio_rows))
print()
for N, size in crossover.items():
    if size is None:
        print(f"N = {N}: the counter index is not faster at the largest size")
    elif size == SIZES[0]:
        print(f"N = {N}: the counter index searched faster (sum over servers) at every measured size")
    else:
        print(f"N = {N}: the counter index searched faster (sum over servers) from {size} files on")

N  files   paper / counter index, slowest server  paper / counter index, sum over servers
-  ------  -------------------------------------  ---------------------------------------
3  100     150.8                                  152.9
3  1000    154.2                                  155.1
3  10000   137.9                                  149.0
3  100000  175.2                                  183.1
5  100     129.1                                  87.5
5  1000    132.3                                  88.2
5  10000   108.4                                  73.6
5  100000  156.6                                  105.2
7  100     102.5                                  57.1
7  1000    114.4                                  56.1
7  10000   122.9                                  54.7
7  100000  138.2                                  78.3

N = 3: the counter index searched faster (sum over servers) at every measured size
N = 5: the counter index searched faster (sum over servers) at every me

## Conditions after the run, and the checks

The cell fails the run unless every timed run returned the right files, every counter-index server did
exactly $c_w = 3$ lookups, and the paper's servers scanned every stored item.

In [6]:
load_end = snapshot()
print(busy_summary(load_start, load_end, PIN_CPU))
print(f"kernel main thread affinity: {list(current_affinity())}")
failures = sum(m.failures for m in results.values())
runs = sum(len(m.runs) + 1 for m in results.values())
print(f"failures over all {runs} checked runs: {failures}")
lookups = {key: list(m.wire) for key, m in results.items() if key[2] == "counter index"}
scanned = {key: list(m.wire) for key, m in results.items() if key[2] == "paper"}
print(f"lookups per server in every counter index run: {sorted({n for w in lookups.values() for n in w})}")

assert failures == 0, "every run returned exactly the 3 matching files"
assert all(w == [len(EXPECTED)] * key[0] for key, w in lookups.items()), "each server did c_w lookups at every size"
assert all(w == [key[1]] for key, w in scanned.items()), "the paper's servers scanned every stored item"
assert all(sum(items for items, _ in held[(N, size)]) == size and sum(rows for _, rows in held[(N, size)]) == size for N, size in held), "every size was reached in both schemes"
assert current_affinity() == pin.affinity, "affinity unchanged since the first cell"
print("all search scaling checks passed")

busy share over the whole run: CPU 2 (the kernel) 100%, CPU 3 (its hyperthread sibling) 0%
kernel main thread affinity: [2]
failures over all 144 checked runs: 0
lookups per server in every counter index run: [3]
all search scaling checks passed
